# Train the U-Net rim segmenter on a Colab GPU

The code lives in the repository; this notebook only fetches it, unpacks the tiles and runs
`lunar_crater.training.train_unet`. Results are written straight to Google Drive, so they
survive a disconnected session.

**Before running**

1. *Runtime → Change runtime type → T4 GPU.*
2. Make the tile zip locally and upload it to `MyDrive/lunar_crater/` in Google Drive:
   `python scripts/pack_tiles.py data/interim/tiles/v1 data/processed/lunar_tiles_v1.zip`
3. Set `BRANCH` below if the code is still on a feature branch.

With the default settings, training takes roughly an hour on a T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/lunar_crater"
ZIP_NAME = "lunar_tiles_v1.zip"
ZIP_SHA256 = "da77e11db8513d901bc12d34cfccf6817af496c4ae9bbebea113aeb84265857d"
BRANCH = "master"
RUN_NAME = "unet_v1"
EPOCHS = 40

In [ ]:
REPO = "https://github.com/aditya-patra1011/lunar_crater_detection.git"
!git clone --quiet --depth 1 --branch {BRANCH} {REPO} /content/repo
%pip install --quiet -e /content/repo
!git -C /content/repo log --oneline -1

In [ ]:
import hashlib
import zipfile
from pathlib import Path

zip_path = Path(DRIVE_DIR) / ZIP_NAME
sha = hashlib.sha256(zip_path.read_bytes()).hexdigest()
assert sha == ZIP_SHA256, f"zip checksum {sha} does not match; re-upload it"
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall("/content/tiles")
tiles_dir = Path("/content/tiles") / ZIP_NAME.removeprefix("lunar_tiles_").removesuffix(".zip")
print(tiles_dir, len(list((tiles_dir / "tiles").glob("*.npz"))), "tiles")

In [ ]:
out_dir = f"{DRIVE_DIR}/{RUN_NAME}"
train = f"python -m lunar_crater.training.train_unet {tiles_dir} --out {out_dir} --epochs {EPOCHS}"
!cd /content/repo && {train}

In [ ]:
import pandas as pd

history = pd.read_csv(f"{out_dir}/history.csv")
ax = history.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(6, 3.5))
ax.set_ylabel("loss")
print("best epoch:", int(history.loc[history["val_loss"].idxmin(), "epoch"]))

## Afterwards

Download `best.pt`, `config.json` and `history.csv` from `MyDrive/lunar_crater/unet_v1/` into
`models/unet_v1/` in the local repository. They are scored there with the evaluation harness,
exactly like the classical baseline.